In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect to the local MCP server running in api:mcp-dev
# The server is the FastMCP instance from app/main_mcp.py
client = MultiServerMCPClient(
    {
        "inkwell_mcp": {
            "transport": "stdio",
            "command": "python",
            "args": ["-m", "app.main_mcp"],
            "env": {
                "PYTHONPATH": "/Users/Ilia_Naryshkin/projects/inkwell/api",
                "INKWELL_ENV": "development",
            },
        }
    }
)

In [ ]:
# Get tools and resources from the MCP server
tools = await client.get_tools()

# Get resources
resources = await client.get_resources("inkwell_mcp")

# Get prompts via the session interface
async with client.session("inkwell_mcp") as session:
    prompts_result = await session.list_prompts()

print("=== TOOLS ===")
for tool in tools:
    print(f"{tool.name}")

print("\n=== RESOURCES ===")
for resource in resources:
    print(f"{resource}")

print("\n=== PROMPTS ===")
print("Prompts in result:")
for prompt in prompts_result.prompts:
    print(f"{prompt.name}")

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    tools=tools,
)

print("Agent created successfully!")

In [ ]:
from langchain.messages import HumanMessage

# Test the agent with a query
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Let me know if Inkwell is online")]}
)

In [ ]:
for msg in response["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")